# Partial Pipeline V1 — End-to-End Visual Debugging

This notebook follows the current execution order in `backend.image_processor.process_image()`. It exposes intermediate state without running the pipeline twice.

In [ ]:
import base64
import io
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "examples" else cwd
if not (PROJECT_ROOT / "backend").is_dir():
    raise RuntimeError(
        "Run this notebook from the repository root or examples directory."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.config import config
from backend.core.helpers import _calc_kernel_size, _image_to_base64
from backend.core.layerd_refine import expand_mask
from backend.core.occlusion import assign_pair_roles
from backend.image_processor import (
    ProcessResult,
    _apply_pair_decisions,
    _build_reconstruction_masks,
    _complete_overlapping_objects,
    _extract_object_layers,
    _extract_objects,
    _generate_final_background,
    _link_overlap_partners,
    _refine_masks,
)

logging.basicConfig(level=logging.INFO)
ALPHA_THRESHOLD = 0.005

def show_mask_overlay(ax, source_image, mask, title, color=(1.0, 0.2, 0.2)):
    ax.imshow(source_image)
    binary = np.asarray(mask) > 0
    overlay = np.zeros((*binary.shape, 4), dtype=np.float32)
    overlay[..., :3] = color
    overlay[..., 3] = binary.astype(np.float32) * 0.45
    ax.imshow(overlay)
    ax.set_title(title)
    ax.axis("off")

def decode_png_base64(value):
    return Image.open(io.BytesIO(base64.b64decode(value))).copy()


## Model configuration and initialization policy

The production `ModelManager` uses lazy loading. This cell reports the selected device and adapters without initializing all GPU models at once. SAM3 loads during extraction; SDAmodal/DIFT loads only when cross-class overlap exists; BiRefNet and inpainting load at their own stages.

In [ ]:
model_categories = ("segmentation", "completion", "matting", "inpainting")
print("Selected device:", config.device)
for category in model_categories:
    model_config = config.get_model_config(category)
    print(f"{category:>12}: {model_config['name']}")
print("Models remain lazy and load at their first processing stage.")


## Data loading

Set the input image and semantic prompts here. Missing inputs fail explicitly; a synthetic fallback image would hide segmentation and overlap problems.

In [ ]:
image_path = PROJECT_ROOT / "assets" / "images" / "composite.png"
keywords = ["men", "women"]

keywords = [keyword.strip() for keyword in keywords if keyword.strip()]
if not keywords:
    raise ValueError("keywords must contain at least one non-empty prompt")
if not image_path.is_file():
    raise FileNotFoundError(f"Input image not found: {image_path}")

with Image.open(image_path) as source:
    image = source.convert("RGB")
image_np = np.asarray(image, dtype=np.uint8)
width, height = image.size

plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.title(f"Source — {width}×{height} — prompts: {keywords}")
plt.axis("off")
plt.show()


## Step 1 — SAM3 extraction and same-class grouping

`_extract_objects()` runs text-guided segmentation, merges overlapping masks within each semantic class, and computes one modal bounding box for every grouped object. If no object is found, production returns the original image; this debugging notebook stops so later plots do not use empty state.

In [ ]:
objects = _extract_objects(image, keywords)
if not objects:
    raise RuntimeError(
        "No objects detected. Production would return the original background."
    )

print(f"Grouped objects: {len(objects)}")
for detected in objects:
    modal_area = int(np.count_nonzero(detected.modal_mask))
    print(
        detected.object_id,
        f"class={detected.semantic_class!r}",
        f"label={detected.display_label!r}",
        f"area={modal_area}",
        f"bbox={detected.bbox}",
    )


### Step 1 visualization — modal masks and grouped boxes

Each row shows the grouped modal mask and the same support over the source image. Red rectangles are the bounding boxes used by overlap detection.

In [ ]:
fig, axes = plt.subplots(len(objects), 2, figsize=(12, 5 * len(objects)), squeeze=False)
for row, detected in enumerate(objects):
    axes[row, 0].imshow(detected.modal_mask, cmap="gray")
    axes[row, 0].set_title(f"{detected.display_label} — modal mask")
    axes[row, 0].axis("off")

    show_mask_overlay(
        axes[row, 1], image, detected.modal_mask, detected.display_label
    )
    x, y, box_width, box_height = detected.bbox
    axes[row, 1].add_patch(
        Rectangle(
            (x, y), box_width, box_height, fill=False, edgecolor="yellow", linewidth=2
        )
    )
plt.tight_layout()
plt.show()


## Step 2 — cross-class overlap graph

Overlap is checked after same-class grouping. Only bounding boxes from different semantic classes with positive intersection area become partners and trigger completion.

In [ ]:
overlap_pairs = _link_overlap_partners(objects)
objects_by_id = {detected.object_id: detected for detected in objects}
print(f"Cross-class overlap pairs: {len(overlap_pairs)}")
for pair in overlap_pairs:
    print("pair:", pair)
for detected in objects:
    print(detected.object_id, "partners:", sorted(detected.overlap_partner_ids))

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image)
colors = plt.cm.tab10(np.linspace(0, 1, max(len(objects), 1)))
for color, detected in zip(colors, objects):
    x, y, box_width, box_height = detected.bbox
    ax.add_patch(Rectangle((x, y), box_width, box_height, fill=False, edgecolor=color, linewidth=2))
    ax.text(x, max(0, y - 4), detected.display_label, color=color, fontsize=10, weight="bold")

for first_id, second_id in overlap_pairs:
    first = objects_by_id[first_id]
    second = objects_by_id[second_id]
    fx, fy, fw, fh = first.bbox
    sx, sy, sw, sh = second.bbox
    ix, iy = max(fx, sx), max(fy, sy)
    ir = min(fx + fw, sx + sw)
    ib = min(fy + fh, sy + sh)
    ax.add_patch(Rectangle((ix, iy), ir - ix, ib - iy, facecolor="magenta", alpha=0.25, edgecolor="magenta"))
ax.set_title("Grouped boxes; magenta = positive-area cross-class intersection")
ax.axis("off")
plt.show()


## Step 3 — conditional SDAmodal completion

Only unique objects with overlap partners are completed. For each completed object, the new hole is `amodal AND NOT modal`; non-overlapping objects keep their modal masks and do not initialize SDAmodal/DIFT.

In [ ]:
_complete_overlapping_objects(image, objects)
completed_objects = [detected for detected in objects if detected.amodal_mask is not None]

if not completed_objects:
    print("No cross-class overlap: SDAmodal/DIFT was skipped.")
else:
    fig, axes = plt.subplots(len(completed_objects), 4, figsize=(18, 4.5 * len(completed_objects)), squeeze=False)
    for row, detected in enumerate(completed_objects):
        axes[row, 0].imshow(detected.modal_mask, cmap="gray")
        axes[row, 0].set_title(f"{detected.display_label} — modal")
        axes[row, 1].imshow(detected.amodal_mask, cmap="gray")
        axes[row, 1].set_title("amodal")
        axes[row, 2].imshow(detected.completion_hole_mask, cmap="magma")
        axes[row, 2].set_title(f"hole — {detected.completion_hole_area} px")
        show_mask_overlay(axes[row, 3], image, detected.completion_hole_mask, "hole overlay", color=(1.0, 0.0, 1.0))
        for column in range(3):
            axes[row, column].axis("off")
    plt.tight_layout()
    plt.show()


## Step 4 — pairwise occluded/occluder decisions

For every overlap edge, the object with the larger completion-hole area is marked occluded. Equal areas are ambiguous and add no occluder assignment.

In [ ]:
hole_areas = {
    detected.object_id: detected.completion_hole_area
    for detected in objects
    if detected.completion_hole_area is not None
}
pair_decisions = assign_pair_roles(overlap_pairs, hole_areas)
_apply_pair_decisions(objects, pair_decisions)

if not pair_decisions:
    print("No overlap pairs require a role decision.")
for decision in pair_decisions:
    first_area = hole_areas[decision.first_id]
    second_area = hole_areas[decision.second_id]
    if decision.ambiguous:
        role_text = "ambiguous"
    else:
        role_text = f"occluded={decision.occluded_id}, occluder={decision.occluder_id}"
    print(
        f"{decision.first_id}({first_area}) vs {decision.second_id}({second_area}): {role_text}"
    )


## Step 5 — reconstruction masks (diagnostic boundary)

These masks identify where hidden RGB reconstruction should occur. **Hidden RGB reconstruction is not implemented yet.** The current production path computes these masks but does not consume them; BiRefNet below still receives the original image and each object's modal mask.

In [ ]:
kernel_size = _calc_kernel_size(image_np)
_build_reconstruction_masks(objects, kernel_size)
reconstruction_objects = [
    detected for detected in objects if detected.reconstruction_mask is not None
]
print("Kernel size:", kernel_size)

if not reconstruction_objects:
    print("No decisive occluded object requires a reconstruction mask.")
else:
    fig, axes = plt.subplots(len(reconstruction_objects), 2, figsize=(12, 5 * len(reconstruction_objects)), squeeze=False)
    for row, detected in enumerate(reconstruction_objects):
        axes[row, 0].imshow(detected.reconstruction_mask, cmap="gray")
        axes[row, 0].set_title(f"{detected.display_label} — reconstruction mask")
        axes[row, 0].axis("off")
        show_mask_overlay(axes[row, 1], image, detected.reconstruction_mask, "diagnostic overlay", color=(0.1, 0.8, 1.0))
    plt.tight_layout()
    plt.show()


## Step 6 — current BiRefNet matting path

This mirrors the current production implementation exactly: `raw_masks` are modal masks, and BiRefNet processes the original RGB image. Amodal support will enter this stage only after hidden RGB reconstruction is integrated.

In [ ]:
raw_masks = [detected.modal_mask for detected in objects]
labels = [detected.display_label for detected in objects]
soft_alphas = _refine_masks(image_np, raw_masks)

fig, axes = plt.subplots(len(soft_alphas), 2, figsize=(12, 5 * len(soft_alphas)), squeeze=False)
for row, (label, alpha) in enumerate(zip(labels, soft_alphas)):
    axes[row, 0].imshow(alpha, cmap="gray", vmin=0.0, vmax=1.0)
    axes[row, 0].set_title(f"{label} — alpha [{alpha.min():.4f}, {alpha.max():.4f}]")
    axes[row, 0].axis("off")
    show_mask_overlay(axes[row, 1], image, alpha > ALPHA_THRESHOLD, f"{label} — alpha support", color=(0.2, 1.0, 0.3))
plt.tight_layout()
plt.show()


## Step 7 — RGBA object-layer extraction

`_extract_object_layers()` uses the configured inpainting model to estimate a component background, refines foreground color and alpha, and returns cropped base64 PNG layers with full-image offsets.

In [ ]:
layers = _extract_object_layers(
    image, image_np, soft_alphas, labels, kernel_size
)
decoded_layers = [decode_png_base64(layer.png_base64) for layer in layers]

if not layers:
    print("No non-empty RGBA layers were produced.")
else:
    fig, axes = plt.subplots(1, len(layers), figsize=(5 * len(layers), 5), squeeze=False)
    for column, (layer, layer_image) in enumerate(zip(layers, decoded_layers)):
        axes[0, column].imshow(layer_image)
        axes[0, column].set_title(
            f"{layer.keyword}\nxy=({layer.x}, {layer.y}), size={layer.width}×{layer.height}"
        )
        axes[0, column].axis("off")
        print(layer.keyword, "offset=", (layer.x, layer.y), "size=", (layer.width, layer.height))
    plt.tight_layout()
    plt.show()


## Step 8 — final background removal and inpainting

The first three views reproduce the production removal-union calculation for diagnostics: modal coverage plus soft-alpha support, then kernel expansion. The production helper is called once to create and refine the final background from the original image.

In [ ]:
visible_union = np.logical_or.reduce([mask > 0 for mask in raw_masks])
for alpha in soft_alphas:
    visible_union |= alpha > ALPHA_THRESHOLD
expanded_union = expand_mask(visible_union, kernel_size).astype(bool)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(visible_union, cmap="gray")
axes[0].set_title("Visible union")
axes[0].axis("off")
axes[1].imshow(expanded_union, cmap="gray")
axes[1].set_title("Expanded inpainting mask")
axes[1].axis("off")
show_mask_overlay(axes[2], image, expanded_union, "Final removal overlay")
plt.tight_layout()
plt.show()

background = _generate_final_background(
    image, raw_masks, soft_alphas, kernel_size
)
plt.figure(figsize=(8, 8))
plt.imshow(background)
plt.title("Final inpainted background")
plt.axis("off")
plt.show()


## Step 9 — assemble the production-shaped result

The final result is assembled from values already computed above. Calling `process_image()` here would repeat all model inference and increase GPU memory pressure.

In [ ]:
result = ProcessResult(
    background_base64=_image_to_base64(background),
    original_width=width,
    original_height=height,
    layers=layers,
)
print(
    f"ProcessResult: {result.original_width}×{result.original_height}, "
    f"{len(result.layers)} layers"
)
for layer in result.layers:
    print(
        f"- {layer.keyword}: bbox=({layer.x}, {layer.y}, {layer.width}, {layer.height})"
    )


## Optional output export

Set `EXPORT_OUTPUTS = True` to save the final background and decoded RGBA layers. Export is disabled by default so rerunning the notebook does not write files unexpectedly.

In [ ]:
EXPORT_OUTPUTS = False
output_dir = PROJECT_ROOT / "examples" / "out" / "partial_pipeline_v1"

if EXPORT_OUTPUTS:
    output_dir.mkdir(parents=True, exist_ok=True)
    background.save(output_dir / "background.png")
    for index, (layer, layer_image) in enumerate(zip(layers, decoded_layers)):
        safe_label = layer.keyword.replace("/", "_").replace("\\", "_")
        layer_image.save(output_dir / f"layer_{index}_{safe_label}.png")
    print(f"Saved outputs to {output_dir}")
else:
    print("Export disabled; set EXPORT_OUTPUTS = True to save files.")
